# Production Agent Architecture

## Scenario: durable EU checkout investigation

Northstar accepts a tenant-scoped request through a gateway, routes it to a session-aware orchestrator, checkpoints an asynchronous worker while evidence is pending, then resumes safely after worker loss. **Safety boundary:** the runtime produces proposals only; identity, policy, tool authorization, and approval are application controls.

![Production agent architecture](../../../assets/production-agent-architecture.svg)

Gateways and workers scale and are replaceable. Run state, queue/schedule metadata, checkpoints, audit records, session continuity, policy decisions, and scoped memory live in durable systems. This separation enables recovery and horizontal scaling.

## 1. Design boundaries

The gateway authenticates, validates, rate-limits, and attaches correlation/tenant IDs. The orchestrator selects an architecture/session and creates a durable run. The runtime takes bounded steps. Stores preserve state/checkpoints; queues schedule asynchronous work; MCP and RAG mediate tools/knowledge; policy/identity controls every transition; observability/evaluation measure quality and operations. Cache keys include tenant, authorization, freshness, source, and policy/catalog version.

In [ ]:
"""Credential-free production architecture simulator: gateway, queue, checkpoint, and recovery."""
from dataclasses import dataclass, field
@dataclass
class Run: run_id:str; tenant:str; status:str="received"; attempts:int=0; checkpoint:str=""; trace:list[str]=field(default_factory=list)
def gateway(run:Run, authenticated:bool=True)->bool:
 if not authenticated: run.status="blocked"; run.trace.append("gateway:block"); return False
 run.trace.append("gateway:accepted"); return True
def enqueue(run:Run): run.status="queued"; run.trace.append("queue:enqueued")
def worker_step(run:Run, external_ready:bool)->str:
 run.attempts+=1
 if not external_ready: run.checkpoint="waiting-evidence"; run.status="waiting"; run.trace.append("checkpoint:waiting-evidence"); return "wait"
 run.checkpoint="proposal-ready"; run.status="complete"; run.trace.append("checkpoint:proposal-ready"); return "complete"
def recover(run:Run): run.trace.append(f"recover:{run.checkpoint}"); return run
def run_demo()->Run:
 run=Run("r-1","acme"); assert gateway(run); enqueue(run); assert worker_step(run,False)=="wait"; recover(run); assert worker_step(run,True)=="complete"; return run
if __name__=="__main__": print(run_demo())


In [1]:
from pathlib import Path
if not (TOPIC / 'lab.py').exists():

run = Run('r-1', 'acme')
assert gateway(run)
enqueue(run)
assert worker_step(run, external_ready=False) == 'wait'
print('checkpoint:', run.checkpoint, run.trace)
recover(run)
assert worker_step(run, external_ready=True) == 'complete'
print('final:', run.status, run.trace)

checkpoint: waiting-evidence ['gateway:accepted', 'queue:enqueued', 'checkpoint:waiting-evidence']
final: complete ['gateway:accepted', 'queue:enqueued', 'checkpoint:waiting-evidence', 'recover:waiting-evidence', 'checkpoint:proposal-ready']


## 2. Durable workflows and operations

Checkpoint after durable transitions and revalidate scope, policy, freshness, budget, event provenance, and idempotency on resume. Use queues/schedules/authenticated events rather than polling models. Classify transient failures, use bounded backoff/jitter, send terminal failures to a dead-letter queue, and make writes idempotent/reconcilable. Autoscale stateless workers from queue depth/service time while protecting upstream tools with concurrency limits/circuit breakers.

For disaster recovery, define RPO/RTO, backup/restore, encryption keys, region failover, state migration, replay safety, degraded read-only operation, and recovery drills.

In [2]:
# Deliberate failure: requests blocked at the gateway must never create queued work.
blocked = Run('r-blocked', 'globex')
assert not gateway(blocked, authenticated=False)
assert blocked.status == 'blocked'
print(blocked.trace)
telemetry = {'gateway_ms': 20, 'queue_ms': 180, 'runtime_ms': 420, 'tool_ms': 610, 'recovery_ms': 35, 'total_ms': 1265}
assert telemetry['total_ms'] >= sum(telemetry[k] for k in ('gateway_ms','queue_ms','runtime_ms','tool_ms'))
telemetry

['gateway:block']


{'gateway_ms': 20,
 'queue_ms': 180,
 'runtime_ms': 420,
 'tool_ms': 610,
 'recovery_ms': 35,
 'total_ms': 1265}

## Production checklist and exercises

- Separate stateless gateways/workers from durable session/run, queue, checkpoint, audit, and memory stores.
- Bound tokens/actions/retries/queue age/concurrency/spend/runtime/fan-out; provide cancellation, escalation, and dead-letter handling.
- Enforce tenant/identity/policy at gateway, cache, store, queue, MCP, RAG, and tool boundaries.
- Test crash/recovery, duplicates, partial writes, stale cache, schema migrations, dependency outage, rate limits, queue replay, failover, and restore.

**Exercises:** add an idempotency key, design a cache key, implement a retry/DLQ policy, specify SLO alerts, and write an RPO/RTO recovery test.

References: [LangGraph durable execution](https://docs.langchain.com/oss/python/langgraph/durable-execution), [Temporal workflows](https://docs.temporal.io/workflows), [MCP authorization](https://modelcontextprotocol.io/extensions/auth/enterprise-managed-authorization).